# 6. Batch effects and confounding: is the model learning biology or tissue?

**Learning goals**: detect a batch effect, remove it without removing real biology, and evaluate
within groups so a model cannot score by learning group labels.

Two traps in this data:
1. **Batch.** Expression comes from two pipelines (Sanger and Broad). The source can be predicted
   from the profiles almost perfectly.
2. **Tissue.** Blood-cancer lines respond to many drugs more than solid-tumour lines do. A model
   that only recognises "blood vs solid" from expression can post a good overall correlation.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(7)

In [2]:
from drugresponse import data
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
E = data.load_expression(); M = data.load_models().reindex(E.index)
s = (M.expression_source == "Sanger").values.astype(float)
T = pd.get_dummies(M.tissue).values.astype(float)
A = np.c_[np.ones(len(s)), T]
sr = s - A @ np.linalg.lstsq(A, s, rcond=None)[0]          # source, net of tissue
Xr = E.values.astype(float)
beta = (sr @ Xr) / (sr @ sr)                               # each gene's source effect
Xc = Xr - np.outer(s, beta)
for name, XX in [("raw", Xr), ("batch-corrected", Xc)]:
    v = XX.var(0); Z = XX[:, np.argsort(v)[-3000:]]; Z = (Z - Z.mean(0)) / Z.std(0)
    P = PCA(20, random_state=0).fit_transform(Z)
    a_src = cross_val_score(LogisticRegression(max_iter=4000), P, s, cv=5, scoring="roc_auc").mean()
    print(f"{name:<16} source predictable from 20 PCs: AUC {a_src:.3f}")

raw              source predictable from 20 PCs: AUC 0.999


batch-corrected  source predictable from 20 PCs: AUC 0.520


The correction regresses each gene on [source, tissue] and subtracts only the source part.
Regressing on source alone would also remove tissue differences wherever the two sources sampled
tissues unevenly. In the project the correction is estimated inside each training fold.

## Evaluating beyond tissue

In [3]:
from drugresponse.experiment import Store, run_drug
from drugresponse.evaluate import within_tissue_rho
from drugresponse.models import TissueModel, RidgeModel
S = Store()
d = S.drugs.index[S.drugs.drug_name == "Cytarabine"][0]
p, _ = run_drug(S, d, [TissueModel, RidgeModel], keep_coefs=False)
for m, g in p.groupby("model"):
    wr = within_tissue_rho(g, S.M.tissue) if m != "Tissue" else float("nan")
    print(f"{m:<7} overall r = {np.corrcoef(g.pred, g.actual)[0, 1]:.2f}   within-tissue Spearman = {wr:.2f}")

Ridge   overall r = 0.70   within-tissue Spearman = 0.39
Tissue  overall r = 0.52   within-tissue Spearman = nan


The tissue-only model already reaches a respectable overall correlation for cytarabine (blood
cancers are far more sensitive). The right question for an expression model is how much it adds
to that, and how well it ranks lines *within* a cancer type. That is why the project reports the
gain over the tissue model and the within-tissue Spearman, and why it tests cancer types held out
entirely (notebooks/05).

**Key takeaways**
* Check whether a technical variable is predictable from your features; if it is, correct for it.
* Correct for batch while protecting the biology you care about (regress on both).
* Always compare with a baseline built from the confounder alone.